In [ ]:
import pandas as pd
import seaborn as sns
import tifffile
from matplotlib import pyplot as plt
import numpy as np
import math
import os
import torch
from tqdm import tqdm

# Load H&E Image

In [ ]:
data_dir = "PATH/TO/DATA/"
res_dir = "PATH/TO/RESULT/"
image = tifffile.imread(f"{res_dir}he-raw.tiff")

In [ ]:
print(image.shape)
plt.imshow(image, cmap='binary')
plt.title('Level 3 H&E', size=16)
plt.axis('scaled')
plt.show()

# Load Transcript Coordinates

In [ ]:
transcriptsurl = f"{data_dir}outs/transcripts.parquet"
transcripts = pd.read_parquet(transcriptsurl)
transcripts['feature_name'] = transcripts['feature_name']
transcripts

# Obtain Transcript Coordinates for Downstream Analysis

## Subset Transcripts

Extract high-quality transcripts and convert micrometer-scale coordinates to pixel-scale coordinates.

In [ ]:
transcripts = transcripts[~transcripts['feature_name'].astype(str).str.startswith(tuple(('NegControlCodeword', 'BLANK', 'NegControlProbe','antisense', 'UnassignedCodeword')))]
transcripts_hq = transcripts.loc[transcripts['qv'] >= 20].copy()
transcripts_hq['x_location'] = transcripts_hq['x_location'].apply(lambda x: x / 1.7)
transcripts_hq['y_location'] = transcripts_hq['y_location'].apply(lambda y: y / 1.7)

In [ ]:
transcripts_hq = transcripts_hq.copy()
print("Number of gene:\t\t" + str(len(np.unique(transcripts_hq['feature_name']))))
print("Number of transcript:\t" + str(len(transcripts_hq)))

## Save Transcript Coordinates to TSV file

In [ ]:
transcripts_use = transcripts_hq.copy()

In [ ]:
mydata1 = transcripts_use[['feature_name', 'x_location', 'y_location']]
mydata2 = mydata1[~mydata1['x_location'].isna() & ~mydata1['y_location'].isna()] 

In [ ]:
mydata2.to_csv(f"{res_dir}transcripts_use.tsv", sep='\t', index=False)

# Simulate Spot

We will simulate the spot arrangement of 10x Visium, i.e. the spot diameter is 55 um and the center distance is 100 um.

In [ ]:
transcripts = pd.read_csv(
    f"{res_dir}transcripts_use.tsv",
    sep='\t',
    header=0
)

In [ ]:
def gap_real_cal(resolution, simulation):
    if simulation == "proportional":
        return 100 / 55 * resolution
    elif simulation == "fixed_spotDistance":
        return 100
    elif simulation == "fixed_spotSize":
        return resolution
    elif simulation == "dense":
        return resolution
    else:
        raise ValueError("Illegal simulation type, please select one from [Proportional, Fixed_spotDistance, Fixed_spotSize, Dense]")

def spots_simulate_gpu(simulation, resolution, transcripts, output_dir):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    gap_real = gap_real_cal(resolution, simulation)
    if simulation == "fixed_spotSize":
        dir_prefix = "spotDistance_"
        spot_size_real = 55
        print(f"start gap: {resolution}")
    else:
        dir_prefix = "spotSize_"
        spot_size_real = resolution
        print(f"start spot_size: {resolution}")
        
    output_dir_full = f'{output_dir}/{simulation}/{dir_prefix}{resolution}/'
    os.makedirs(output_dir_full, exist_ok = True)
    
    gap = gap_real / 1.7
    start_gap = gap / 2
    radius = spot_size_real / 2 / 1.7
    
    #  spot locations
    spots = pd.DataFrame(data=None, columns=['x','y'])
    min_x = transcripts['x_location'].min() + gap
    min_y = transcripts['y_location'].min() + gap
    max_x = transcripts['x_location'].max() - gap
    max_y = transcripts['y_location'].max() - gap
    
    xs = [ 
        np.arange(np.floor(min_x), max_x, gap), 
        np.arange(np.floor(min_x) + start_gap, max_x, gap) 
    ]
    ys = np.arange(np.floor(min_y), max_y, start_gap * math.sqrt(3))
    
    for i in range(len(ys)): 
        myrow = xs[i % 2]
        tmp = pd.DataFrame({'x': myrow, 'y': np.repeat(ys[i], len(myrow))})
        spots = pd.concat([spots, tmp], ignore_index=True)
        
    spots = spots.sort_values('x').reset_index(drop=True)
    spots.index = 'barcode' + spots.index.astype(str)
    spots['spot'] = spots.index
        
    spots.iloc[:,[2,0,1]].to_csv(f'{output_dir_full}/locs-raw.tsv', sep='\t', index=False)
    
    print(f"start cnts of {simulation} with {resolution}")
    
    spot_xy = torch.tensor(spots[['x', 'y']].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    
    transcript_xy = torch.tensor(transcripts[['x_location', 'y_location']].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    transcript_gene = transcripts['feature_name'].astype('category')
    gene_ids = torch.tensor(transcript_gene.cat.codes.values, device=device)
    all_gene_names = transcript_gene.cat.categories.tolist()
    cnts = torch.zeros((spot_xy.shape[0], len(all_gene_names)), dtype=torch.int32, device=device)
    for i in tqdm(range(spot_xy.shape[0])):
        center = spot_xy[i]  # shape [2]
        distances = torch.norm(transcript_xy - center, dim=1)  # [num_transcripts]
        mask = distances <= radius
        selected_genes = gene_ids[mask].long()  
        if selected_genes.numel() > 0:
            cnts[i].scatter_add_(0, selected_genes, torch.ones_like(selected_genes, dtype=torch.int32))
    cnts_df = pd.DataFrame(cnts.cpu().numpy(), index=spots.index, columns=all_gene_names)
    cnts_df.to_csv(f'{output_dir_full}/cnts.tsv', sep='\t', index=True, index_label='spot')
    
    print(f"finished {simulation} with {resolution}")


In [ ]:
spot_sizes = {
    'dense':list(range(20, 155, 5)),
    'proportional':list(range(20, 155, 5)),
    'fixed_spotSize':list(range(55, 205, 5)),
    'fixed_spotDistance':list(range(20, 105, 5))
}
spot_sizes

In [ ]:
output_dir = f"{data_dir}/raw_resolution/matrix/"
for simulation in spot_sizes.keys():
    for resolution in spot_sizes[simulation]:
        spots_simulate_gpu(simulation, resolution, transcripts, output_dir)